# Test: `/api/explain-image` EndpointThis notebook tests the **`POST /api/explain-image`** endpoint which accepts a fresh imageand a label, then returns binary concept explanations — no prior `/api/classify` call needed.**Test image:** `demo/samples/n02325366_6190.JPEG`

## 0. Configuration

In [ ]:
import requestsimport jsonimport base64import iofrom pathlib import Pathfrom PIL import Imageimport matplotlib.pyplot as pltimport numpy as npAPI_BASE = "http://localhost:8501"IMAGE_PATH = Path("samples/n02325366_6190.JPEG")LABEL = "rabbit"  # change this to test different labelsassert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH}"print(f"Image : {IMAGE_PATH}")print(f"Label : {LABEL}")print(f"API   : {API_BASE}")

## 1. Health Check

In [ ]:
resp = requests.get(f"{API_BASE}/api/health")resp.raise_for_status()print("Health check:", resp.json())

## 2. Visualize Input Image

In [ ]:
input_img = Image.open(IMAGE_PATH).convert("RGB")fig, ax = plt.subplots(figsize=(6, 6))ax.imshow(input_img)ax.set_title(f"Input: {IMAGE_PATH.name}", fontsize=13, fontweight="bold")ax.axis("off")plt.tight_layout()plt.show()print(f"Size: {input_img.size[0]}x{input_img.size[1]}")

## 3. Call `/api/explain-image`

In [ ]:
with open(IMAGE_PATH, "rb") as f:    files = {"file": (IMAGE_PATH.name, f, "image/jpeg")}    data  = {"label": LABEL}    resp  = requests.post(f"{API_BASE}/api/explain-image", files=files, data=data)resp.raise_for_status()result = resp.json()print(f"Status       : {resp.status_code}")print(f"Image ID     : {result.get('image_id')}")print(f"Model Output : {result.get('model_output', '')[:200]}")print(f"# Concepts   : {len(result.get('top_concepts', []))}")print(f"Mask colors  : {result.get('mask_colors_hex')}")print(f"Bbox colors  : {result.get('bbox_colors_hex')}")

## 4. Raw JSON Response

In [ ]:
# Print the full response (truncate large base64 strings for readability)def _truncate_b64(obj, max_len=80):    if isinstance(obj, dict):        return {k: _truncate_b64(v, max_len) for k, v in obj.items()}    elif isinstance(obj, list):        return [_truncate_b64(v, max_len) for v in obj]    elif isinstance(obj, str) and len(obj) > max_len and ";base64," in obj[:60]:        return obj[:60] + "...<truncated>"    return objprint(json.dumps(_truncate_b64(result), indent=2))

## 5. Visualize Top ConceptsFor each returned concept we show:- **Rank** & **similarity** score- **Concept names** and **predictions**- **Prototype images** (with and without mask overlay)

In [ ]:
def b64_to_pil(b64_str):    if "," in b64_str:        b64_str = b64_str.split(",", 1)[1]    return Image.open(io.BytesIO(base64.b64decode(b64_str)))concepts = result.get("top_concepts", [])mask_colors = result.get("mask_colors_hex", ["#ff0000"] * len(concepts))if not concepts:    print("No concepts returned.")else:    for i, concept in enumerate(concepts):        rank = concept["rank"]        sim  = concept["similarity"]        names = ", ".join(concept.get("concept_names", [])) or "—"        preds = ", ".join(concept.get("predictions", [])) or "—"        protos = concept.get("prototypes", [])        color  = mask_colors[i % len(mask_colors)]        print(f"\n{'='*60}")        print(f"  Concept #{rank}  |  similarity = {sim:.4f}  |  color = {color}")        print(f"  Names      : {names}")        print(f"  Predictions: {preds}")        print(f"  Prototypes : {len(protos)}")        print(f"{'='*60}")        if not protos:            continue        n_protos = len(protos)        fig, axes = plt.subplots(2, n_protos, figsize=(4 * n_protos, 5))        if n_protos == 1:            axes = axes.reshape(2, 1)        for j, proto in enumerate(protos):            img_masked = b64_to_pil(proto["image_b64"])            axes[0, j].imshow(img_masked)            axes[0, j].set_title(f"Prototype {j+1} (masked)", fontsize=10)            axes[0, j].axis("off")            clean_key = proto.get("image_b64_clean", proto["image_b64"])            img_clean = b64_to_pil(clean_key)            axes[1, j].imshow(img_clean)            axes[1, j].set_title(f"Prototype {j+1} (clean)", fontsize=10)            axes[1, j].axis("off")        fig.suptitle(            f"Concept #{rank}: {names}  (sim={sim:.3f})",            fontsize=12, fontweight="bold", y=1.02,        )        plt.tight_layout()        plt.show()

## 6. Summary — Input vs. Explanation Side-by-Side

In [ ]:
concepts = result.get("top_concepts", [])proto_imgs = []proto_labels = []for c in concepts:    if c.get("prototypes"):        proto_imgs.append(b64_to_pil(c["prototypes"][0]["image_b64"]))        name = ", ".join(c.get("concept_names", [])) or f"Concept {c['rank']}"        proto_labels.append(f"#{c['rank']} {name}\nsim={c['similarity']:.3f}")n_cols = 1 + len(proto_imgs)fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))axes[0].imshow(input_img)axes[0].set_title(f"Input\n{IMAGE_PATH.name}", fontsize=10, fontweight="bold")axes[0].axis("off")for idx, (img, lbl) in enumerate(zip(proto_imgs, proto_labels)):    axes[idx + 1].imshow(img)    axes[idx + 1].set_title(lbl, fontsize=9)    axes[idx + 1].axis("off")fig.suptitle(    f'explain-image(label="{LABEL}") — Top {len(proto_imgs)} Concepts',    fontsize=13, fontweight="bold", y=1.04,)plt.tight_layout()plt.show()

## 7. Compare: `/api/classify` then `/api/explain` vs. `/api/explain-image`Run the traditional two-step flow for comparison.

In [ ]:
# Step 1: classifywith open(IMAGE_PATH, "rb") as f:    classify_resp = requests.post(        f"{API_BASE}/api/classify",        files={"file": (IMAGE_PATH.name, f, "image/jpeg")},    )classify_resp.raise_for_status()classify_data = classify_resp.json()print("Classify response:")print(f"  image_id     : {classify_data['image_id']}")print(f"  model_output : {classify_data['model_output']}")print(f"  nouns        : {classify_data['nouns']}")print(f"  prompt       : {classify_data['prompt']}")# Step 2: explain using the image_idexplain_resp = requests.post(    f"{API_BASE}/api/explain",    json={"image_id": classify_data["image_id"], "label": LABEL},)explain_resp.raise_for_status()explain_data = explain_resp.json()print(f"\nExplain (two-step) — # concepts: {len(explain_data.get('top_concepts', []))}")for c in explain_data.get("top_concepts", []):    print(f"  #{c['rank']}  sim={c['similarity']:.4f}  names={c.get('concept_names', [])}")print(f"\nExplain-image (one-step) — # concepts: {len(result.get('top_concepts', []))}")for c in result.get("top_concepts", []):    print(f"  #{c['rank']}  sim={c['similarity']:.4f}  names={c.get('concept_names', [])}")

---### API Summary| Endpoint | Method | Input | Output ||---|---|---|---|| `/api/explain` | POST | `{"image_id": "...", "label": "..."}` (JSON) | top concepts + prototypes || `/api/explain-image` | POST | `file` + `label` (multipart/form-data) | top concepts + prototypes + `image_id` |The **explain-image** endpoint is a convenience wrapper that combines image upload + explanation in a single call.